# Rerank existing text retrieval results with Llama Nemotron

This is the simple, fast starting point: it uses a few synthetic text chunks representing results from an existing retriever, reranks them with `nvidia/llama-nemotron-rerank-vl-1b-v2`, and measures the change in ranking quality. It does not require the NeMo Retriever Library, an index, or an ingestion pipeline. An optional second example uses Nemotron 3 embeddings to produce the candidates before applying the same reranker; optional embedding retrieval or local model downloads can take longer. For a full, longer-running text-only evaluation of one real dataset, see [Rerank ViDoRe v3 HR](reranking_vidore_v3_hr_text_only.ipynb).

A reranker only reorders the candidates it receives. Retrieve more candidates than you ultimately return so the reranker has useful choices, and evaluate the candidate recall separately.

## Configure inference

`INFERENCE_MODE="hosted"` calls NVIDIA-hosted endpoints with Python's standard library and requires `NVIDIA_API_KEY`, but does not require a local GPU. The notebook checks the environment variable before inference and securely prompts when it is missing. `INFERENCE_MODE="local"` runs the same VL reranker with Transformers and requires a suitable GPU. This example sends text-only query and passage inputs; the model also supports visual inputs. NVIDIA's [API reference](https://docs.api.nvidia.com/nim/reference/nvidia-llama-nemotron-rerank-vl-1b-v2-infer) documents the hosted endpoint.

Set `RUN_EMBEDDING_RETRIEVAL=False` when you only want the reranker-first example.

In [1]:
INFERENCE_MODE = "hosted"  # "hosted" or "local"
RUN_EMBEDDING_RETRIEVAL = True
TOP_K = 5
RERANK_BATCH_SIZE = 8
RERANK_MAX_LENGTH = 512

RERANK_MODEL = "nvidia/llama-nemotron-rerank-vl-1b-v2"
RERANK_INVOKE_URL = (
    "https://ai.api.nvidia.com/v1/retrieval/nvidia/"
    "llama-nemotron-rerank-vl-1b-v2/reranking"
)
EMBED_MODEL = {
    "local": "nvidia/Nemotron-3-Embed-1B-BF16",
    "hosted": "nvidia/nemotron-3-embed-1b",
}[INFERENCE_MODE]
EMBED_INVOKE_URL = "https://integrate.api.nvidia.com/v1/embeddings"

assert INFERENCE_MODE in {"local", "hosted"}
assert TOP_K > 0
assert RERANK_BATCH_SIZE > 0 and RERANK_MAX_LENGTH > 0

## Dependencies

Hosted mode needs only a Jupyter Python kernel and an NVIDIA API key. The optional installation cell below does nothing unless you uncomment a `%pip` command.

For optional local mode, use a GPU-enabled kernel with a working NVIDIA driver. Install matching PyTorch and torchvision builds for its CUDA environment, then Transformers. The [VL reranker model card](https://huggingface.co/nvidia/llama-nemotron-rerank-vl-1b-v2) requires `transformers>=4.56.0` and shows direct sequence-classification inference through its `AutoProcessor`; this reranker does **not** need Sentence Transformers. If you also run the optional [Nemotron 3 Embed](https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16) section, use `transformers>=5.2.0` and `sentence-transformers>=5.4.1`. A clean kernel environment avoids mixing system and user-site packages. Use the [PyTorch installation selector](https://pytorch.org/get-started/locally/) for matching builds. If you change packages, restart the kernel. A missing `ipywidgets` progress bar is only a warning; `operator torchvision::nms does not exist` indicates incompatible PyTorch and torchvision builds.

The first opt-in repair command is for **CUDA 12.4 only**, matching a `torch==2.5.1+cu124` installation. Uncomment it only if that is your intended CUDA wheel; otherwise choose a matching command from [PyTorch's version-specific instructions](https://docs.pytorch.org/get-started/previous-versions/). The second command installs Transformers for local reranking and also satisfies the optional embedding model's minimum. Uncomment the third command only for optional local embeddings. If a pre-existing old `datasets` installation fails to import `HfFolder` from `huggingface_hub`, use the fourth command to upgrade `datasets`; the reranker does not require it. `%pip` installs into the selected notebook kernel, not the Jupyter server. Restart the kernel after running any command and rerun the notebook from the top. Installing packages cannot provide a missing GPU or NVIDIA driver; if an incompatibility persists, use a clean GPU-enabled kernel instead.

In [2]:
# Optional CUDA 12.4 repair: uncomment the next line only when needed.
#%pip install "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu124

# Local VL reranker (also satisfies the optional embedding model's Transformers minimum).
#%pip install --upgrade "transformers>=5.2.0,<6"

# Optional local Nemotron 3 Embed dependency.
#%pip install --upgrade "sentence-transformers>=5.4.1"

# Optional repair for an older installed datasets package importing removed HfFolder.
#%pip install --upgrade "datasets>=4.8.2,<5"

In [3]:
from importlib.util import find_spec

if INFERENCE_MODE == "local":
    required = ("torch", "transformers")
    missing = [package for package in required if find_spec(package) is None]
    if missing:
        raise ImportError(
            f"Local inference requires {', '.join(missing)}. "
            "Install the optional local dependencies described above."
        )
    from importlib.metadata import version
    from packaging.version import Version

    transformers_version = version("transformers")
    if Version(transformers_version) < Version("4.56.0"):
        raise RuntimeError(
            f"Local VL reranking needs transformers>=4.56.0; found "
            f"{transformers_version}. Run the optional Transformers %pip command "
            "in Dependencies, then restart the kernel."
        )
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Local inference requires a CUDA GPU visible to this kernel. "
            "Use INFERENCE_MODE='hosted' or select a GPU-enabled kernel."
        )

## Prepare credentials and imports

For hosted mode, start Jupyter from a shell where `NVIDIA_API_KEY` is exported. If the variable is absent, the notebook prompts without displaying the key and sets it only in the current kernel process environment. It never stores the key in the notebook.

In [4]:
import json
import math
import os
from getpass import getpass
from urllib.request import Request, urlopen

from IPython.display import Markdown, display


def require_nvidia_api_key():
    api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if not api_key:
        api_key = getpass("NVIDIA API key: ").strip()
    if not api_key:
        raise EnvironmentError(
            "NVIDIA_API_KEY is required when INFERENCE_MODE='hosted'."
        )
    os.environ["NVIDIA_API_KEY"] = api_key
    return api_key


def post_json(url, payload):
    request = Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {require_nvidia_api_key()}",
            "Accept": "application/json",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urlopen(request, timeout=120) as response:
        return json.load(response)


def show_table(rows, columns):
    def cell(value):
        text = f"{value:.3f}" if isinstance(value, float) else str(value)
        return text.replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    lines.extend(
        "| " + " | ".join(cell(row[column]) for column in columns) + " |"
        for row in rows
    )
    display(Markdown("\n".join(lines)))


if INFERENCE_MODE == "hosted":
    require_nvidia_api_key()
    print("NVIDIA_API_KEY is available to this notebook kernel.")

## Start with post-retrieval candidates

The following fixture represents chunks returned by another retriever. Each candidate retains its original score and rank. The `relevance` labels are only for evaluating this worked example; production retriever output does not need them. Replace each `candidates` list with results from your own retriever.

The examples intentionally place strong answers below weaker lexical matches so the before-and-after comparison is visible.

In [5]:
evaluation_examples = [
    {
        "query": "Can I get a refund if I cancel a flight within 24 hours?",
        "candidates": [
            {
                "chunk_id": "travel-insurance",
                "text": (
                    "Travel insurance can reimburse covered cancellations after "
                    "the insurer reviews a claim."
                ),
                "retrieval_score": 0.94,
                "relevance": 0,
            },
            {
                "chunk_id": "flight-changes",
                "text": (
                    "Passengers can change many tickets before departure, subject "
                    "to fare differences and change fees."
                ),
                "retrieval_score": 0.91,
                "relevance": 1,
            },
            {
                "chunk_id": "twenty-four-hour-rule",
                "text": (
                    "For flights booked at least seven days before departure, you can "
                    "cancel within 24 hours of booking for a full refund."
                ),
                "retrieval_score": 0.88,
                "relevance": 3,
            },
            {
                "chunk_id": "nonrefundable-ticket",
                "text": (
                    "Nonrefundable tickets normally return no cash after the grace "
                    "period, although the airline can issue a travel credit."
                ),
                "retrieval_score": 0.84,
                "relevance": 2,
            },
            {
                "chunk_id": "baggage-refund",
                "text": "Baggage fees are refunded when the airline loses a checked bag.",
                "retrieval_score": 0.81,
                "relevance": 0,
            },
        ],
    },
    {
        "query": "What should I do if my laptop battery is swelling?",
        "candidates": [
            {
                "chunk_id": "battery-life",
                "text": "Lower screen brightness to extend laptop battery life.",
                "retrieval_score": 0.95,
                "relevance": 0,
            },
            {
                "chunk_id": "battery-calibration",
                "text": "Battery calibration can improve an inaccurate charge estimate.",
                "retrieval_score": 0.90,
                "relevance": 0,
            },
            {
                "chunk_id": "swollen-battery-safety",
                "text": (
                    "Stop using and charging a device with a swollen lithium-ion "
                    "battery. Power it down and arrange safe professional removal."
                ),
                "retrieval_score": 0.86,
                "relevance": 3,
            },
            {
                "chunk_id": "battery-disposal",
                "text": (
                    "Take damaged lithium-ion batteries to an approved recycling or "
                    "hazardous-waste facility; do not put them in household trash."
                ),
                "retrieval_score": 0.82,
                "relevance": 2,
            },
            {
                "chunk_id": "laptop-cleaning",
                "text": "Use compressed air to remove dust from laptop vents.",
                "retrieval_score": 0.78,
                "relevance": 0,
            },
        ],
    },
]

for example in evaluation_examples:
    for rank, candidate in enumerate(example["candidates"], start=1):
        candidate["original_rank"] = rank

## Rerank the candidates

`rerank_candidates` accepts ordinary result dictionaries and preserves their metadata. In hosted mode, it calls the NVIDIA reranking endpoint directly and maps each returned score to its original candidate index. In local mode, it follows the VL reranker's model-card recipe: prepare each text-only `question`/`doc_text` pair with the model's `AutoProcessor`, then read the sequence-classification logit. The local example uses a 512-token cap and batches eight passages at a time; adjust `RERANK_MAX_LENGTH` and `RERANK_BATCH_SIZE` to fit your GPU and chunk lengths. It adds `_rerank_score` and sorts by that score. If an upstream stage already added `_rerank_score`, the adapter preserves it as `previous_reranker_score`.

In [6]:
local_reranker = None
local_processor = None
if INFERENCE_MODE == "local":
    from transformers import AutoModelForSequenceClassification, AutoProcessor

    local_processor = AutoProcessor.from_pretrained(
        RERANK_MODEL, trust_remote_code=True, rerank_max_length=RERANK_MAX_LENGTH
    )
    rerank_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    local_reranker = AutoModelForSequenceClassification.from_pretrained(
        RERANK_MODEL, trust_remote_code=True, torch_dtype=rerank_dtype,
        attn_implementation="sdpa",
    ).eval().to("cuda")


def local_rerank_scores(query, passages):
    scores = []
    for start in range(0, len(passages), RERANK_BATCH_SIZE):
        batch = passages[start : start + RERANK_BATCH_SIZE]
        examples = [
            {"question": query, "doc_text": passage, "doc_image": ""}
            for passage in batch
        ]
        inputs = local_processor.process_queries_documents_crossencoder(examples)
        inputs = {
            name: value.to("cuda") if isinstance(value, torch.Tensor) else value
            for name, value in inputs.items()
        }
        with torch.inference_mode():
            logits = local_reranker(**inputs, return_dict=True).logits
        scores.extend(float(score) for score in logits.view(-1).cpu().tolist())
    return scores


def hosted_rerank_scores(query, passages):
    result = post_json(
        RERANK_INVOKE_URL,
        {
            "model": RERANK_MODEL,
            "query": {"text": query},
            "passages": [{"text": passage} for passage in passages],
            "truncate": "END",
        },
    )
    scores = [None] * len(passages)
    for ranking in result["rankings"]:
        index = int(ranking["index"])
        if not 0 <= index < len(scores) or scores[index] is not None:
            raise ValueError("Reranker returned an invalid candidate index.")
        scores[index] = float(ranking["logit"])
    if any(score is None or not math.isfinite(score) for score in scores):
        raise ValueError("Reranker did not return one finite score per candidate.")
    return scores


def rerank_candidates(query, candidates, top_n=None):
    if not candidates:
        return []
    hits = []
    for candidate in candidates:
        hit = dict(candidate)
        if "_rerank_score" in hit:
            hit["previous_reranker_score"] = hit.pop("_rerank_score")
        hits.append(hit)
    passages = [str(hit["text"]) for hit in hits]
    if INFERENCE_MODE == "local":
        scores = local_rerank_scores(query, passages)
    else:
        scores = hosted_rerank_scores(query, passages)
    if len(scores) != len(hits) or any(not math.isfinite(score) for score in scores):
        raise ValueError("Reranker must return one finite score per candidate.")
    for hit, score in zip(hits, scores):
        hit["_rerank_score"] = score
    ranked = sorted(hits, key=lambda hit: hit["_rerank_score"], reverse=True)
    return ranked[:top_n] if top_n is not None else ranked

/home/edwardk/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████████████| 601/601 [00:00<00:00, 3187.39it/s]


In [7]:
rankings = []
for example in evaluation_examples:
    original = [dict(candidate) for candidate in example["candidates"]]
    reranked = rerank_candidates(example["query"], original)
    rankings.append(
        {
            "query": example["query"],
            "original": original,
            "reranked": reranked,
        }
    )

for result in rankings:
    comparison = [
        {
            "reranked_rank": rank,
            "original_rank": hit["original_rank"],
            "chunk_id": hit["chunk_id"],
            "retrieval_score": hit["retrieval_score"],
            "reranker_score": hit["_rerank_score"],
            "relevance": hit["relevance"],
            "text": hit["text"],
        }
        for rank, hit in enumerate(result["reranked"], start=1)
    ]
    print(result["query"])
    show_table(comparison, list(comparison[0]))

Can I get a refund if I cancel a flight within 24 hours?


| reranked_rank | original_rank | chunk_id | retrieval_score | reranker_score | relevance | text |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 3 | twenty-four-hour-rule | 0.880 | 4.490 | 3 | For flights booked at least seven days before departure, you can cancel within 24 hours of booking for a full refund. |
| 2 | 4 | nonrefundable-ticket | 0.840 | -2.137 | 2 | Nonrefundable tickets normally return no cash after the grace period, although the airline can issue a travel credit. |
| 3 | 2 | flight-changes | 0.910 | -4.054 | 1 | Passengers can change many tickets before departure, subject to fare differences and change fees. |
| 4 | 1 | travel-insurance | 0.940 | -4.323 | 0 | Travel insurance can reimburse covered cancellations after the insurer reviews a claim. |
| 5 | 5 | baggage-refund | 0.810 | -4.707 | 0 | Baggage fees are refunded when the airline loses a checked bag. |

What should I do if my laptop battery is swelling?


| reranked_rank | original_rank | chunk_id | retrieval_score | reranker_score | relevance | text |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 3 | swollen-battery-safety | 0.860 | -0.295 | 3 | Stop using and charging a device with a swollen lithium-ion battery. Power it down and arrange safe professional removal. |
| 2 | 1 | battery-life | 0.950 | -3.305 | 0 | Lower screen brightness to extend laptop battery life. |
| 3 | 4 | battery-disposal | 0.820 | -3.388 | 2 | Take damaged lithium-ion batteries to an approved recycling or hazardous-waste facility; do not put them in household trash. |
| 4 | 5 | laptop-cleaning | 0.780 | -3.669 | 0 | Use compressed air to remove dust from laptop vents. |
| 5 | 2 | battery-calibration | 0.900 | -5.660 | 0 | Battery calibration can improve an inaccurate charge estimate. |

## Measure ranking quality

Normalized Discounted Cumulative Gain (nDCG) rewards relevant chunks more when they appear near the top. The graded labels in this fixture range from `0` for irrelevant to `3` for a direct answer. A positive delta means the reranker improved the ordering for these candidates.

Reranking does not change candidate recall because it cannot add a missing chunk.

In [8]:
def ndcg_at_k(hits, k):
    relevances = [float(hit["relevance"]) for hit in hits[:k]]
    dcg = sum(
        (2**relevance - 1) / math.log2(rank + 1)
        for rank, relevance in enumerate(relevances, start=1)
    )
    ideal = sorted(
        (float(hit["relevance"]) for hit in hits),
        reverse=True,
    )[:k]
    idcg = sum(
        (2**relevance - 1) / math.log2(rank + 1)
        for rank, relevance in enumerate(ideal, start=1)
    )
    return dcg / idcg if idcg else 0.0


quality_rows = []
for result in rankings:
    before = ndcg_at_k(result["original"], TOP_K)
    after = ndcg_at_k(result["reranked"], TOP_K)
    quality_rows.append(
        {
            "query": result["query"],
            f"nDCG@{TOP_K} before": before,
            f"nDCG@{TOP_K} after": after,
            "delta": after - before,
        }
    )

quality = quality_rows + [
    {
        "query": "Mean",
        f"nDCG@{TOP_K} before": sum(
            row[f"nDCG@{TOP_K} before"] for row in quality_rows
        ) / len(quality_rows),
        f"nDCG@{TOP_K} after": sum(
            row[f"nDCG@{TOP_K} after"] for row in quality_rows
        ) / len(quality_rows),
        "delta": sum(row["delta"] for row in quality_rows) / len(quality_rows),
    }
]
show_table(quality, list(quality[0]))

| query | nDCG@5 before | nDCG@5 after | delta |
| --- | --- | --- | --- |
| Can I get a refund if I cancel a flight within 24 hours? | 0.577 | 1.000 | 0.423 |
| What should I do if my laptop battery is swelling? | 0.539 | 0.956 | 0.417 |
| Mean | 0.558 | 0.978 | 0.420 |

## Add or replace reranking in an existing system

To **add** reranking, pass the ordered dictionaries from your retriever to `rerank_candidates`. Use `top_n` only after reranking.

To **replace** an existing reranker, disable the upstream reranking stage and request a larger raw candidate set from the embedding or lexical retriever. Preserve the chunk text and identifiers when adapting its output. Applying multiple rerankers is possible, but each stage adds cost and later stages cannot recover candidates that an earlier stage removed.

Retriever scores and reranker scores have different meanings. Compare ranks or evaluated quality instead of comparing the numeric scores directly.

## Optional: produce candidates with embeddings

This section retains the embedding path while keeping the reranker as the focus. It embeds the first example's small candidate corpus, performs cosine-similarity retrieval, and sends the top candidates to the same `rerank_candidates` adapter. In an application, replace this in-memory search with results from your existing vector database. Local Nemotron 3 Embed needs `transformers>=5.2.0` and `sentence-transformers>=5.4.1`; the separate opt-in `%pip` commands in Dependencies install them. Sentence Transformers is only used here, not by the reranker. Hosted mode does not need these packages.

In [9]:
def hosted_embeddings(texts, input_type):
    result = post_json(
        EMBED_INVOKE_URL,
        {
            "input": texts,
            "model": EMBED_MODEL,
            "input_type": input_type,
            "encoding_format": "float",
            "truncate": "END",
        },
    )
    rows = sorted(result["data"], key=lambda row: row["index"])
    if [row["index"] for row in rows] != list(range(len(texts))):
        raise ValueError("Embedding response does not match the input texts.")
    return [row["embedding"] for row in rows]


def cosine_similarity(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return dot / (left_norm * right_norm)


if RUN_EMBEDDING_RETRIEVAL:
    embedding_example = evaluation_examples[0]
    corpus = [dict(candidate) for candidate in embedding_example["candidates"]]
    documents = [candidate["text"] for candidate in corpus]

    if INFERENCE_MODE == "local":
        from packaging.version import Version
        import transformers
        try:
            import sentence_transformers
        except ImportError as exc:
            if "HfFolder" in str(exc) and "huggingface_hub" in str(exc):
                raise RuntimeError(
                    "An old datasets package is incompatible with huggingface_hub. "
                    "Run the optional datasets %pip command in Dependencies, "
                    "then restart the kernel."
                ) from None
            raise

        minimum_versions = {
            "transformers": "5.2.0",
            "sentence-transformers": "5.4.1",
        }
        installed_versions = {
            "transformers": transformers.__version__,
            "sentence-transformers": sentence_transformers.__version__,
        }
        outdated = [
            f"{name}=={installed_versions[name]} (need >={minimum})"
            for name, minimum in minimum_versions.items()
            if Version(installed_versions[name]) < Version(minimum)
        ]
        if outdated:
            raise RuntimeError(
                "Local Nemotron 3 embedding dependencies are too old: "
                + ", ".join(outdated)
                + ". Run the optional Transformers and Sentence Transformers "
                "%pip commands in Dependencies, then restart the kernel."
            )

        from sentence_transformers import SentenceTransformer

        embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
        document_vectors = embedder.encode_document(documents).tolist()
        query_vector = embedder.encode_query(
            [embedding_example["query"]]
        )[0].tolist()
        del embedder
    else:
        document_vectors = hosted_embeddings(documents, "passage")
        query_vector = hosted_embeddings(
            [embedding_example["query"]],
            "query",
        )[0]

    similarities = [
        cosine_similarity(query_vector, vector) for vector in document_vectors
    ]
    candidate_order = sorted(
        range(len(corpus)), key=lambda index: similarities[index], reverse=True
    )[:TOP_K]
    embedding_candidates = []
    for original_rank, index in enumerate(candidate_order, start=1):
        candidate = dict(corpus[index])
        candidate["retrieval_score"] = similarities[index]
        candidate["original_rank"] = original_rank
        embedding_candidates.append(candidate)

    embedding_reranked = rerank_candidates(
        embedding_example["query"],
        embedding_candidates,
        top_n=TOP_K,
    )
    embedding_quality = [
        {
            "ranking": "Embedding retrieval",
            f"nDCG@{TOP_K}": ndcg_at_k(embedding_candidates, TOP_K),
        },
        {
            "ranking": "Embedding retrieval + reranking",
            f"nDCG@{TOP_K}": ndcg_at_k(embedding_reranked, TOP_K),
        },
    ]
    show_table(embedding_quality, list(embedding_quality[0]))
    show_table(
        embedding_reranked,
        ["original_rank", "chunk_id", "retrieval_score", "_rerank_score", "relevance", "text"],
    )
else:
    print("Embedding retrieval skipped. Set RUN_EMBEDDING_RETRIEVAL=True to run it.")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Loading weights: 100%|████████████████████████████████████████| 146/146 [00:00<00:00, 3458.05it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


| ranking | nDCG@5 |
| --- | --- |
| Embedding retrieval | 0.946 |
| Embedding retrieval + reranking | 1.000 |

| original_rank | chunk_id | retrieval_score | _rerank_score | relevance | text |
| --- | --- | --- | --- | --- | --- |
| 1 | twenty-four-hour-rule | 0.668 | 4.490 | 3 | For flights booked at least seven days before departure, you can cancel within 24 hours of booking for a full refund. |
| 3 | nonrefundable-ticket | 0.443 | -2.137 | 2 | Nonrefundable tickets normally return no cash after the grace period, although the airline can issue a travel credit. |
| 5 | flight-changes | 0.283 | -4.054 | 1 | Passengers can change many tickets before departure, subject to fare differences and change fees. |
| 2 | travel-insurance | 0.452 | -4.323 | 0 | Travel insurance can reimburse covered cancellations after the insurer reviews a claim. |
| 4 | baggage-refund | 0.408 | -4.707 | 0 | Baggage fees are refunded when the airline loses a checked bag. |

## Tradeoffs when you add a reranker

- **Quality and recall:** A cross-encoder can improve candidate ordering, but it cannot recover relevant chunks that the first-stage retriever did not return. Measure first-stage recall and reranked nDCG separately.
- **Candidate depth:** More candidates give the reranker more opportunities to find the best chunks, but increase inference cost. Tune the retrieval depth and final `top_n` together.
- **Latency and throughput:** A reranker scores every query-candidate pair. Batching improves throughput, while smaller candidate sets reduce latency.
- **Input length:** Long chunks can be truncated. Align chunk size and the model's input limit so the evidence that determines relevance is retained.
- **Score interpretation:** Reranker scores order candidates for one query; they are not calibrated probabilities and should not be compared across queries without validation.
- **Model fit:** Evaluate the reranker on your domains, languages, and modalities. This notebook uses text chunks, while the selected model also supports visual inputs.
- **Deployment:** Hosted inference reduces local GPU requirements but sends inputs to an endpoint and adds network dependency. Local inference keeps data in your environment but requires model storage, GPU memory, and operational capacity.